<a href="https://colab.research.google.com/github/Munjiwon/SpecialTopics-in-TextMining/blob/master/ch02/bm25_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2주차 실습 2 — BM25 직접 구현

**이 노트북의 새 개념**: TF 포화(k₁)와 문서 길이 정규화(b)가 순위를 어떻게 바꾸는지 본다.

score(q, d) = Σ idf(w) · f(w,d)(k₁+1) / (f(w,d) + k₁(1 − b + b·|d|/avgdl))

모든 문서의 점수를 한 번에 계산하도록 PyTorch 텐서로 구현한다.

In [1]:
import sys, torch
print("Python", sys.version.split()[0], "| torch", torch.__version__)

Python 3.13.15 | torch 2.11.0+cpu


In [8]:
# 말뭉치 올리기 — 1주차와 같은 파일을 쓴다(빈 줄로 구분된 문단 하나를 문서 하나로 본다)
CORPUS_PATH = "/content/corpus.txt"      # Colab 밖에서 실행할 때는 이 경로를 직접 바꾼다
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     CORPUS_PATH = "/content/" + next(iter(uploaded))
# except ImportError:
#     pass

with open(CORPUS_PATH, encoding="utf-8") as f:
    raw = f.read()
docs = [d.strip() for d in raw.split("\n") if len(d.strip()) > 20]   # 문단 = 문서
print(f"문서 {len(docs):,}개, 예시: {docs[0][:60]}...")

문서 73,098개, 예시: 0	돌겠네 진짜. 황숙아, 어크 공장 그만 돌려라. 죽는다....


In [9]:
from collections import Counter
import math

tokenized = [d.split() for d in docs]
N = len(tokenized)
lengths = torch.tensor([len(t) for t in tokenized], dtype=torch.float32)    # |d|
avgdl = lengths.mean()
df = Counter(w for t in tokenized for w in set(t))

def idf(w):
    # 음수가 되지 않는 BM25 idf — log(1 + (N − df + 0.5) / (df + 0.5))
    n = df.get(w, 0)
    return math.log(1 + (N - n + 0.5) / (n + 0.5))

def bm25(query, k1=1.5, b=0.75):
    scores = torch.zeros(N)
    for w in query.split():
        f = torch.tensor([t.count(w) for t in tokenized], dtype=torch.float32)   # 문서별 빈도
        scores += idf(w) * f * (k1 + 1) / (f + k1 * (1 - b + b * lengths / avgdl))
    return scores

query = " ".join(docs[0].split()[:2])            # 첫 문서의 앞 두 단어를 질의로 쓴다
print("질의:", query)
top = torch.topk(bm25(query), min(5, N))
for s, i in zip(top.values, top.indices):
    print(f"{s:6.2f}  (길이 {int(lengths[i]):>4})  {docs[i][:50]}")

질의: 0 돌겠네
 14.46  (길이    6)  0	최적화 언제하냐 할때마다 돌겠네 진짜
 12.84  (길이    9)  0	돌겠네 진짜. 황숙아, 어크 공장 그만 돌려라. 죽는다.
  6.64  (길이   34)  0	뭐 이것저것 다 좋은데 싱글플레이 다 이겨놓으니까 결과화면 가기 직전에 겜 멈춤ㅡㅡ 스
  1.14  (길이   22)  0	작품성 10 동영상 재생성 50 개연성 0 퍼즐 흥미도 0 숨은그림찾기 완성도 0 뜬금
  1.13  (길이    2)  0	재밌어??????어??????????재밌냐고!!!!!!!!!!!!!!!!!!!!!!!!


## k₁·b를 바꿔 순위 변화 보기
b = 0 이면 길이를 무시하므로 긴 문서가 위로 올라온다.

In [10]:
for k1, b in [(1.2, 0.75), (2.0, 0.75), (1.5, 0.0), (1.5, 1.0)]:
    top = torch.topk(bm25(query, k1, b), min(5, N)).indices.tolist()
    print(f"k1={k1:<4} b={b:<5} 상위 문서 {top}  평균 길이 {lengths[top].mean():.0f}")

k1=1.2  b=0.75  상위 문서 [27310, 0, 35866, 1726, 2751]  평균 길이 15
k1=2.0  b=0.75  상위 문서 [27310, 0, 35866, 1726, 2751]  평균 길이 15
k1=1.5  b=0.0   상위 문서 [35866, 27310, 0, 1726, 21539]  평균 길이 19
k1=1.5  b=1.0   상위 문서 [27310, 0, 35866, 747, 3069]  평균 길이 11


## 직접 해 보기
1. `rank_bm25` 패키지(`%pip install -q rank-bm25`)의 BM25Okapi 점수와 비교해 보자. idf 정의가 같은지 확인한다.
2. k₁ 을 100 처럼 크게 두면 BM25 가 TF-IDF 와 비슷해지는 이유를 식으로 설명해 보자.
3. 질의 단어를 하나 더 넣으면 점수가 어떻게 합산되는가?